In [1]:
import os
import glob
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

load_dotenv(override = True)

True

In [3]:
folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob = "**/*.md",
        loader_cls = TextLoader,
        loader_kwargs = {"encoding": "utf-8"}
    )

    folder_docs = loader.load()

    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 306 documents


In [4]:
import re

# Add legal identifiers and titles BEFORE splitting
for doc in documents:
    source = doc.metadata.get("source", "")
    filename = os.path.basename(source)

    identifier = ""
    title = ""

    article_match = re.match(r"article_(\d+)", filename)
    recital_match = re.match(r"recital_(\d+)", filename)
    annex_match = re.match(r"annex_([IVXLCDM]+)", filename)

    if article_match:
        number = int(article_match.group(1))
        identifier = f"Article {number}"

        # Extract the title appearing after "Article N"
        title_match = re.search(
            rf"## Official text\s*\n+\s*Article\s+{number}\s*\n+\s*([^\n]+)",
            doc.page_content
        )

        if title_match:
            title = title_match.group(1).strip()

    elif recital_match:
        identifier = f"Recital {int(recital_match.group(1))}"

    elif annex_match:
        identifier = f"Annex {annex_match.group(1)}"

    doc.metadata["identifier"] = identifier
    doc.metadata["title"] = title


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)


# Add legal identity to EVERY chunk
for chunk in chunks:
    identifier = chunk.metadata.get("identifier", "")
    title = chunk.metadata.get("title", "")

    if title:
        header = f"{identifier} — {title}"
    else:
        header = identifier

    chunk.page_content = (
        f"{header}\n\n"
        f"{chunk.page_content}"
    )


print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 1060 chunks
First chunk:

page_content='Annex I

# Annex I

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

ANNEX I	 
List of Union harmonisation legislation
Section A.' metadata={'source': 'knowledge-base\\annexes\\annex_I.md', 'doc_type': 'annexes', 'identifier': 'Annex I', 'title': ''}


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)

if os.path.exists(db_name):
    Chroma(
        persist_directory = db_name,
        embedding_function = embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    persist_directory = db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

Vectorstore created with 1060 documents


In [6]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(
    limit = 1,
    include = ["embeddings"]
)["embeddings"][0]

dimensions = len(sample_embedding)

print(
    f"There are {count:,} vectors with "
    f"{dimensions:,} dimensions in the vector store"
)

There are 1,060 vectors with 384 dimensions in the vector store


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

llm = ChatOpenAI(
    temperature = 0,
    model_name = MODEL
)

In [9]:
retriever.invoke("When does the EU AI Act apply?")

[Document(id='b6df1204-9a16-4bd8-b22d-936cf4a0d5a1', metadata={'doc_type': 'articles', 'title': 'Entry into force and application', 'source': 'knowledge-base\\articles\\article_113.md', 'identifier': 'Article 113'}, page_content='Article 113 — Entry into force and application\n\n# Article 113\n\n## Source\n\nRegulation (EU) 2024/1689 - Artificial Intelligence Act\n\n## Official text\n\nArticle 113 \nEntry into force and application\nThis Regulation shall enter into force on the twentieth day following that of \nits publication in the Official Journal of the European Union.\nIt shall apply from 2 August 2026. However:\n(a)\t Chapters I and II shall apply from 2 February 2025;\n(b)\t Chapter III Section 4, Chapter V, Chapter VII and Chapter XII and \nArticle 78 shall apply from 2 August 2025, with the exception of \nArticle 101;\n(c)\t Article 6(1) and the corresponding obligations in this Regulation shall \napply from 2 August 2027.\n\nThis Regulation shall be binding in its entirety an

In [10]:
SYSTEM_PROMPT_TEMPLATE = """
You are an assistant answering questions about Regulation (EU) 2024/1689
(the EU Artificial Intelligence Act).

Answer the user's question using only the provided context.

Rules:
1. Base your answer only on the provided EU AI Act context.
2. Do not invent or assume legal provisions that are not supported by the context.
3. At the beginning of the answer, explicitly identify the main relevant legal
   provision using its exact identifier, for example:
   "According to Article 113..."
   "According to Recital 47..."
   "According to Annex III..."
4. If several provisions are relevant, identify the most important one first
   and mention the others where appropriate.
5. Clearly distinguish the general rule from exceptions, conditions, and
   special cases.
6. If the provided context is insufficient to answer the question, say that
   the answer cannot be determined from the provided EU AI Act materials.
7. Do not present the answer as legal advice.

Context:
{context}
"""

In [11]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        f"Source: {doc.metadata['source']}\n{doc.page_content}"
        for doc in docs
    )

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        context = context
    )

    response = llm.invoke([
        SystemMessage(content = system_prompt),
        HumanMessage(content = question)
    ])

    return response.content

In [12]:
answer_question(
    "When does the EU AI Act apply?",
    []
)

'According to Article 113, the EU AI Act shall apply from 2 August 2026. However, there are specific provisions that apply earlier: Chapters I and II from 2 February 2025; Chapter III Section 4, Chapter V, Chapter VII, Chapter XII, and Article 78 from 2 August 2025 (with the exception of Article 101); and Article 6(1) and the corresponding obligations from 2 August 2027.'

In [13]:
docs = retriever.invoke(
    "When does the EU AI Act apply?"
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:800])


--- Result 1 ---
{'doc_type': 'articles', 'source': 'knowledge-base\\articles\\article_113.md', 'title': 'Entry into force and application', 'identifier': 'Article 113'}
Article 113 — Entry into force and application

# Article 113

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

Article 113 
Entry into force and application
This Regulation shall enter into force on the twentieth day following that of 
its publication in the Official Journal of the European Union.
It shall apply from 2 August 2026. However:
(a)	 Chapters I and II shall apply from 2 February 2025;
(b)	 Chapter III Section 4, Chapter V, Chapter VII and Chapter XII and 
Article 78 shall apply from 2 August 2025, with the exception of 
Article 101;
(c)	 Article 6(1) and the corresponding obligations in this Regulation shall 
apply from 2 August 2027.

This Regulation shall be binding in its entirety and directly applicable in all 
Member States. Done at Br

--- Result 2 ---
{'identifi

In [14]:
docs = retriever.invoke(
    "Which AI practices are prohibited under the EU AI Act?"
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:1500])


--- Result 1 ---
{'doc_type': 'articles', 'identifier': 'Article 5', 'source': 'knowledge-base\\articles\\article_005.md', 'title': 'Prohibited AI practices'}
Article 5 — Prohibited AI practices

# Article 5

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

--- Result 2 ---
{'identifier': 'Article 108', 'doc_type': 'articles', 'title': 'Amendments to Regulation (EU) 2018/1139', 'source': 'knowledge-base\\articles\\article_108.md'}
Article 108 — Amendments to Regulation (EU) 2018/1139

Article 108 
Amendments to Regulation (EU) 2018/1139
Regulation (EU) 2018/1139 is amended as follows:
(1)	
in Article 17, the following paragraph is added:
‘3. Without prejudice to paragraph 2, when adopting implementing acts 
pursuant to paragraph 1 concerning Artificial Intelligence systems which 
are safety components within the meaning of Regulation (EU) 2024/1689 
of the European Parliament and of the Council (*), the requirements set 
out in Chapter III, Section

In [15]:
docs = vectorstore.similarity_search(
    "Which AI practices are prohibited under the EU AI Act?",
    k = 10
)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:700])


--- Result 1 ---
{'title': 'Prohibited AI practices', 'identifier': 'Article 5', 'source': 'knowledge-base\\articles\\article_005.md', 'doc_type': 'articles'}
Article 5 — Prohibited AI practices

# Article 5

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

--- Result 2 ---
{'identifier': 'Article 108', 'title': 'Amendments to Regulation (EU) 2018/1139', 'doc_type': 'articles', 'source': 'knowledge-base\\articles\\article_108.md'}
Article 108 — Amendments to Regulation (EU) 2018/1139

Article 108 
Amendments to Regulation (EU) 2018/1139
Regulation (EU) 2018/1139 is amended as follows:
(1)	
in Article 17, the following paragraph is added:
‘3. Without prejudice to paragraph 2, when adopting implementing acts 
pursuant to paragraph 1 concerning Artificial Intelligence systems which 
are safety components within the meaning of Regulation (EU) 2024/1689 
of the European Parliament and of the Council (*), the requirements set 
out in Chapter III, Section

In [16]:
# Diagnostic check: inspect Article 5 chunks
# Not required for the RAG pipeline

article5_chunks = [
    chunk
    for chunk in chunks
    if chunk.metadata.get("identifier") == "Article 5"
]

print("Number of Article 5 chunks:", len(article5_chunks))

for i, chunk in enumerate(article5_chunks, 1):
    print(f"\n--- Article 5 chunk {i} ---")
    print(chunk.page_content[:1500])

Number of Article 5 chunks: 17

--- Article 5 chunk 1 ---
Article 5 — Prohibited AI practices

# Article 5

## Source

Regulation (EU) 2024/1689 - Artificial Intelligence Act

## Official text

--- Article 5 chunk 2 ---
Article 5 — Prohibited AI practices

Article 5 
Prohibited AI practices
1.	
The following AI practices shall be prohibited:
(a)	 the placing on the market, the putting into service or the use of an 
AI system that deploys subliminal techniques beyond a person’s 
consciousness or purposefully manipulative or deceptive techniques, 
with the objective, or the effect of materially distorting the behaviour 
of a person or a group of persons by appreciably impairing their 
ability to make an informed decision, thereby causing them to take 
a decision that they would not have otherwise taken in a manner that 
causes or is reasonably likely to cause that person, another person 
or group of persons significant harm;
(b)	 the placing on the market, the putting into service or the

In [17]:
gr.ChatInterface(
    answer_question,
    type = "messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [18]:
from evaluation import test
from collections import Counter

tests = test.load_tests()

print(f"Number of tests: {len(tests)}")

Number of tests: 20


In [19]:
example = tests[0]

print("Question:", example.question)
print("Category:", example.category)
print("Reference answer:", example.reference_answer)
print("Keywords:", example.keywords)

Question: What is the purpose of the EU AI Act?
Category: direct_fact
Reference answer: According to Article 1, the purpose of the EU AI Act is to improve the functioning of the internal market and promote the uptake of human-centric and trustworthy artificial intelligence while ensuring a high level of protection of health, safety and fundamental rights and supporting innovation.
Keywords: ['Article 1', 'internal market', 'human-centric', 'trustworthy', 'fundamental rights', 'innovation']


In [20]:
Counter(t.category for t in tests)

Counter({'obligation': 9,
         'multi_part': 4,
         'classification': 2,
         'direct_fact': 1,
         'scope': 1,
         'definition': 1,
         'transparency': 1,
         'temporal': 1})

In [21]:
def evaluate_retrieval(test_case):
    docs = retriever.invoke(test_case.question)

    retrieved_text = "\n\n".join(
        doc.page_content for doc in docs
    ).lower()

    found_keywords = [
        keyword
        for keyword in test_case.keywords
        if keyword.lower() in retrieved_text
    ]

    keywords_found = len(found_keywords)
    total_keywords = len(test_case.keywords)

    keyword_coverage = (
        keywords_found / total_keywords * 100
        if total_keywords > 0
        else 0
    )

    return {
        "keywords_found": keywords_found,
        "total_keywords": total_keywords,
        "keyword_coverage": keyword_coverage,
        "found_keywords": found_keywords,
        "docs": docs
    }

In [22]:
retrieval_eval = evaluate_retrieval(example)

print("Keywords found:", retrieval_eval["keywords_found"])
print("Total keywords:", retrieval_eval["total_keywords"])
print("Keyword coverage:", retrieval_eval["keyword_coverage"])
print("Found:", retrieval_eval["found_keywords"])

Keywords found: 3
Total keywords: 6
Keyword coverage: 50.0
Found: ['trustworthy', 'fundamental rights', 'innovation']


In [23]:
from pydantic import BaseModel, Field

In [24]:
class AnswerEval(BaseModel):
    feedback: str = Field(
        description = "Brief explanation of the evaluation"
    )
    accuracy: float = Field(
        description = "Accuracy score from 1 to 5"
    )
    completeness: float = Field(
        description = "Completeness score from 1 to 5"
    )
    relevance: float = Field(
        description = "Relevance score from 1 to 5"
    )

In [25]:
evaluator_llm = llm.with_structured_output(AnswerEval)

In [26]:
def evaluate_answer(test_case):
    answer = answer_question(
        test_case.question,
        []
    )

    evaluation_prompt = f"""
You are evaluating the output of a RAG question-answering system.

Question:
{test_case.question}

Reference answer:
{test_case.reference_answer}

RAG answer:
{answer}

Evaluate the RAG answer on three criteria:

1. Accuracy: Is the answer factually correct compared with the reference answer?
2. Completeness: Does it include the important information from the reference answer?
3. Relevance: Does it directly answer the question without irrelevant information?

Give each criterion a score from 1 to 5.

Also provide a brief explanation.
"""

    evaluation = evaluator_llm.invoke(evaluation_prompt)

    return evaluation, answer

In [27]:
evaluation, answer = evaluate_answer(example)

print("Question:")
print(example.question)

print("\nReference answer:")
print(example.reference_answer)

print("\nRAG answer:")
print(answer)

print("\nEvaluation:")
print(evaluation)

Question:
What is the purpose of the EU AI Act?

Reference answer:
According to Article 1, the purpose of the EU AI Act is to improve the functioning of the internal market and promote the uptake of human-centric and trustworthy artificial intelligence while ensuring a high level of protection of health, safety and fundamental rights and supporting innovation.

RAG answer:
According to Recital 1, the purpose of the EU AI Act is to promote the uptake of human-centric and trustworthy artificial intelligence (AI) while ensuring a high level of protection of health, safety, and fundamental rights as enshrined in the Charter of Fundamental Rights of the European Union. It aims to protect against the harmful effects of AI systems in the Union, support innovation, and ensure the free movement and cross-border development, marketing, and use of AI-based goods and services within the Union.

Evaluation:
feedback='The RAG answer accurately captures the main purpose of the EU AI Act, emphasizing 

In [28]:
print("Feedback:", evaluation.feedback)
print("Accuracy:", evaluation.accuracy)
print("Completeness:", evaluation.completeness)
print("Relevance:", evaluation.relevance)

Feedback: The RAG answer accurately captures the main purpose of the EU AI Act, emphasizing the promotion of trustworthy AI and protection of fundamental rights. It includes most key points from the reference, such as supporting innovation and ensuring high protection levels. The answer is relevant and directly addresses the question.
Accuracy: 4.0
Completeness: 4.0
Relevance: 5.0


In [29]:
import importlib
import implementation.answer

importlib.reload(implementation.answer)

<module 'implementation.answer' from 'c:\\Users\\gokif\\projects\\eu_ai_act_rag\\week5\\implementation\\answer.py'>

In [30]:
import evaluation.eval

importlib.reload(evaluation.eval)

from evaluation.eval import evaluate_retrieval, evaluate_answer

In [31]:
from implementation.answer import embeddings

print(len(embeddings.embed_query("test")))


384


In [32]:
from evaluation import test

In [33]:
tests = test.load_tests()

In [34]:
len(tests)

20

In [35]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

What is the purpose of the EU AI Act?
direct_fact
According to Article 1, the purpose of the EU AI Act is to improve the functioning of the internal market and promote the uptake of human-centric and trustworthy artificial intelligence while ensuring a high level of protection of health, safety and fundamental rights and supporting innovation.
['Article 1', 'internal market', 'human-centric', 'trustworthy', 'fundamental rights', 'innovation']


In [36]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'obligation': 9,
         'multi_part': 4,
         'classification': 2,
         'direct_fact': 1,
         'scope': 1,
         'definition': 1,
         'transparency': 1,
         'temporal': 1})

In [37]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [38]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.08333333333333333, ndcg=0.1781035935540111, keywords_found=3, total_keywords=6, keyword_coverage=50.0)

In [39]:
eval, answer, chunks = evaluate_answer(example)

In [40]:
eval

AnswerEval(feedback='The answer accurately captures the main purpose as stated in the reference, including the promotion of human-centric and trustworthy AI, health and safety protections, and facilitating the internal market. It also adds details about protecting democracy and the rule of law, which are relevant and consistent with the law’s scope. The answer is thorough and covers all key points from the reference, with additional context. It is highly relevant to the question.', accuracy=5.0, completeness=5.0, relevance=5.0, legal_source_accuracy=5.0)

In [41]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The answer accurately captures the main purpose as stated in the reference, including the promotion of human-centric and trustworthy AI, health and safety protections, and facilitating the internal market. It also adds details about protecting democracy and the rule of law, which are relevant and consistent with the law’s scope. The answer is thorough and covers all key points from the reference, with additional context. It is highly relevant to the question.
5.0
5.0
5.0
